# IIT_DroneLearning — Evader RL 학습 (Colab Pro+)

**담당**: 이재왕 (work/evader)  
**목적**: Google Colab Pro+에서 Unity Headless 빌드를 사용하여 Evader 에이전트를 학습한다.

## 사용 전 체크리스트
- [ ] Google Drive에 Unity Headless 빌드 업로드 완료
- [ ] `DRIVE_BASE` 경로 실제 경로로 수정
- [ ] `RUN_ID` 실험 이름 설정
- [ ] `STAGE` 현재 학습 단계 설정 (0~3)
- [ ] GPU 런타임 활성화 확인 (메뉴 > 런타임 > 런타임 유형 변경 > GPU)

---
## 1. 환경 설치

In [ ]:
# Version Lock: mlagents==4.0.0, torch>=2.0.0
!pip install -q mlagents==4.0.0
!pip install -q 'torch>=2.0.0,<3.0.0'
!pip install -q tensorboard

# 버전 확인
import mlagents
import torch
print(f'mlagents: {mlagents.__version__}')
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

---
## 2. Google Drive 마운트 및 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── 경로 설정 (실제 경로로 수정) ────────────────────────────────
DRIVE_BASE     = '/content/drive/MyDrive/IIT_DroneLearning'
REPO_PATH      = '/content/IIT_DroneLearning'       # 레포 클론 위치
BUILD_PATH     = f'{DRIVE_BASE}/builds/EvaderEnv'   # Unity Headless 빌드 경로
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
LOG_DIR        = f'{DRIVE_BASE}/runs'

# 디렉토리 생성
for d in [CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Created: {d}')

# ── 실험 설정 ────────────────────────────────────────────────────
STAGE       = 0       # 현재 Stage (0~3)
EXP_NAME    = 'base'  # 실험 이름
SEED        = 42
INIT_FROM   = None    # warm-start run-id (Stage 전환 시 이전 run-id 입력)
NUM_ENVS    = 4       # 병렬 환경 수 (Colab Pro+ GPU 기준)

RUN_ID = f'evader_s{STAGE}_{EXP_NAME}_seed{SEED}'
print(f'Run ID: {RUN_ID}')

---
## 3. 레포 클론 및 Config 확인

In [ ]:
import subprocess

# 레포 클론 (이미 존재하면 pull)
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/alpha7179/IIT_DroneLearning.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull origin work/evader

!cd {REPO_PATH} && git checkout work/evader && git log --oneline -3

# Config 파일 확인
config_path = f'{REPO_PATH}/python/config/evader_s{STAGE}_base.yaml'
if not os.path.exists(config_path):
    config_path = f'{REPO_PATH}/python/config/evader_s{STAGE}_template.yaml'
print(f'Config: {config_path}')
!cat {config_path}

---
## 4. Unity Headless 빌드 준비

Unity 빌드는 **로컬에서 빌드 후 Drive에 업로드**해야 합니다.

### 로컬 빌드 방법 (한 번만)
```
1. Unity Editor에서 File > Build Settings
2. Platform: Linux (x86_64)
3. Scripting Backend: IL2CPP
4. Server Build 체크 (headless)
5. Build → EvaderEnv
6. 빌드 결과물을 Drive의 builds/ 폴더에 업로드
```

In [ ]:
# 빌드 파일 존재 여부 확인
build_exe = f'{BUILD_PATH}/EvaderEnv.x86_64'
if os.path.exists(build_exe):
    !chmod +x {build_exe}
    print(f'Build found: {build_exe}')
else:
    print(f'Build NOT found: {build_exe}')
    print('Unity Headless 빌드를 Drive에 업로드하세요.')
    print('빌드 없이도 --no-graphics 옵션으로 Editor 연결 학습 가능합니다.')

---
## 5. 학습 실행

In [ ]:
import subprocess, sys

cmd = [
    'mlagents-learn', config_path,
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
    f'--num-envs={NUM_ENVS}',
    '--force',
]

# Unity 빌드 경로가 있으면 추가
if os.path.exists(build_exe):
    cmd += [f'--env={build_exe}']

# Warm-start (Stage 전환 시)
if INIT_FROM:
    cmd += [f'--initialize-from={INIT_FROM}']

print('실행 커맨드:')
print(' '.join(cmd))
print()

# 학습 실행 (백그라운드 실행 권장 — Colab 타임아웃 방지)
# 아래 셀을 실행하면 학습이 시작됩니다
result = subprocess.run(cmd, cwd=REPO_PATH)
print(f'Exit code: {result.returncode}')

---
## 6. TensorBoard 모니터링

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

---
## 7. 체크포인트 Drive 저장

In [ ]:
import shutil
from pathlib import Path

# 학습 결과를 Drive에 복사
src = Path(LOG_DIR) / RUN_ID
dst = Path(CHECKPOINT_DIR) / RUN_ID

if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print(f'Saved: {dst}')
    
    # ONNX export 확인
    onnx_files = list(dst.glob('**/*.onnx'))
    if onnx_files:
        print('ONNX models:')
        for f in onnx_files:
            print(f'  {f.name}')
else:
    print(f'Run directory not found: {src}')

---
## 8. 실험 결과 기록 안내

학습 완료 후 로컬에서 `/log` 커맨드로 `Docs/EXPERIMENTS.md`에 기록하세요.

```
/log evader_s0_base_seed42 "survival=?% goal=?% capture=?% 첫 수렴 확인"
```

또는 직접 `Docs/EXPERIMENTS.md` 테이블에 추가 후:
```bash
git add Docs/EXPERIMENTS.md
git commit -m "[Docs] 실험 결과 기록: evader_s0_base_seed42"
git push origin work/evader
```